# Sloinnte: Predicting Surname Equivalence Between Irish and English Forms
**Milestone 2: Data and Baseline Checkpoint**

**Research question:** Can machine-learning models predict whether two surname forms belong to the same surname cluster in the Gaois Database of Irish-Language Surnames?

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from src.sloinnte_utils import (
    load_forms, add_group_ids, split_train_test,
    make_pairs, add_features, FEATURES,
)

In [4]:
df = load_forms("../data/minimal.xml")
df = add_group_ids(df)
train_df, test_df = split_train_test(df)
train_pairs = add_features(make_pairs(split_df=train_df, seed=0))
test_pairs = add_features(make_pairs(split_df=test_df, seed=1))
print("Forms:", len(df))
print("Groups:", df["group_id"].nunique())
print("Train forms:", len(train_df))
print("Test forms:", len(test_df))
print("Overlap:", len(set(train_df["form"]) & set(test_df["form"])))
print("Train pairs:", len(train_pairs))
print("Test pairs:", len(test_pairs))


Forms: 3164
Groups: 938
Train forms: 2531
Test forms: 633
Overlap: 0
Train pairs: 4784
Test pairs: 1136


These match the Milestone 2 numbers exactly, so all models are compared on the same data, split and pairs.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

def cv_f1(model, train_df, n_splits=5):
    """Mean and std of F1 across GroupKFold folds, rebuilding pairs inside each fold."""
    gkf = GroupKFold(n_splits=n_splits)

    scores = []
    for fold, (tr_i, va_i) in enumerate(gkf.split(train_df, groups=train_df["group_id"])):
        fold_tr = add_features(make_pairs(train_df.iloc[tr_i], seed=fold))
        fold_va = add_features(make_pairs(train_df.iloc[va_i], seed=fold + 100))
        m = clone(model)
        m.fit(fold_tr[FEATURES], fold_tr["is_match"])
        pred = m.predict(fold_va[FEATURES])
        scores.append(f1_score(fold_va["is_match"], pred))

    return np.mean(scores), np.std(scores)

    
    

NameError: name 'mean' is not defined

In [8]:
from sklearn.linear_model import LogisticRegression

mean, std = cv_f1(LogisticRegression(max_iter=1000), train_df)
print(f"{mean:.3f} ± {std:.3f}")


0.862 ± 0.013


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

C_values = [0.01, 0.1, 1, 10, 100]
results = []

for C in C_values:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    mean, std = cv_f1(model, train_df)
    results.append({"model": "Logistic regression", "setting": f"C={C}", "cv_f1_mean": mean, "cv_f1_std": std})

lr_results = pd.DataFrame(results)
lr_results

,model,setting,cv_f1_mean,cv_f1_std
0,Logistic regression,C=0.01,0.854054,0.019387
1,Logistic regression,C=0.1,0.863865,0.009621
2,Logistic regression,C=1,0.863727,0.008754
3,Logistic regression,C=10,0.863821,0.008917
4,Logistic regression,C=100,0.864057,0.008910


Performance is flat for C ≥ 0.1 (CV F1 ≈ 0.864); differences are much smaller than the fold-to-fold standard deviation (≈ 0.009). Only C = 0.01 is clearly worse. C = 0.1 is chosen as the simplest (most regularised) setting within the noise. With only six features, logistic regression is not prone to overfitting, so regularisation strength has little effect.

In [12]:
from sklearn.tree import DecisionTreeClassifier

depths = [2, 3, 4, 5, 6, 7, 8, None]
results = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    mean, std = cv_f1(model, train_df)
    results.append({"model": "Decision Tree", "setting": f"max_depth={depth}", "cv_f1_mean": mean, "cv_f1_std": std})

tree_results = pd.DataFrame(results)
tree_results

,model,setting,cv_f1_mean,cv_f1_std
0,Decision Tree,max_depth=2,0.813183,0.024261
1,Decision Tree,max_depth=3,0.844394,0.012079
2,Decision Tree,max_depth=4,0.846137,0.014805
3,Decision Tree,max_depth=5,0.854297,0.014389
4,Decision Tree,max_depth=6,0.850972,0.022436
5,Decision Tree,max_depth=7,0.848474,0.018149
6,Decision Tree,max_depth=8,0.845442,0.012294
7,Decision Tree,max_depth=None,0.810980,0.013046


CV F1 rises from depth 2 (0.813, underfitting) to a peak at depth 5 (0.854), then falls; the unrestricted tree (0.811) overfits. Using the same rule as for logistic regression (simplest setting within one standard deviation of the best), max_depth = 3 (0.844) is chosen, which also keeps the tree small enough to inspect. Even the best tree is below logistic regression (0.864).

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

ks = [3, 5, 11, 21, 31, 51]
results = []

for k in ks:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    mean, std = cv_f1(model, train_df)
    results.append({"model": "k-NN", "setting": f"n_neighbors={k}", "cv_f1_mean": mean, "cv_f1_std": std})

knn_results = pd.DataFrame(results)
knn_results

,model,setting,cv_f1_mean,cv_f1_std
0,k-NN,n_neighbors=3,0.835431,0.015835
1,k-NN,n_neighbors=5,0.845834,0.016268
2,k-NN,n_neighbors=11,0.855577,0.022111
3,k-NN,n_neighbors=21,0.857777,0.017303
4,k-NN,n_neighbors=31,0.859412,0.016163
5,k-NN,n_neighbors=51,0.855503,0.019900


CV F1 rises from k = 3 (0.835) to a peak at k = 31 (0.859), then falls slightly at k = 51; small k gives noisy, over-complex decision boundaries. Scores for k = 11–51 are within one standard deviation of each other. k = 31 is chosen as the best-scoring setting. Features are standardised inside the pipeline, so the scaler is fitted on each fold's training pairs only.

In [16]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

def evaluate(name, y_true, pred, score):
    """Return test-set metrics for one model as a dictionary."""
    return {
        "model": name,
        "F1": f1_score(y_true, pred),
        "Precision": precision_score(y_true, pred),
        "Recall": recall_score(y_true, pred),
        "ROC-AUC": roc_auc_score(y_true, score), 
    }

In [18]:
from sklearn.dummy import DummyClassifier

X_train, y_train = train_pairs[FEATURES], train_pairs["is_match"]
X_test, y_test = test_pairs[FEATURES], test_pairs["is_match"]

test_results = []

# Dummy
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_score = dummy.predict_proba(X_test)[:, 1]
test_results.append(evaluate("Dummy (stratified)", y_test, dummy_pred, dummy_score))

# Rules baseline (threshold chosen in Milestone 2 CV)
rule_pred = (test_pairs["edit_dist"] <= 0.75).astype(int)
rule_score = 1 - test_pairs["edit_dist"]
test_results.append(evaluate("Rules (edit dist ≤ 0.75)", y_test, rule_pred, rule_score))

pd.DataFrame(test_results).round(3)

,model,F1,Precision,Recall,ROC-AUC
0,Dummy (stratified),0.472,0.473,0.472,0.473
1,Rules (edit dist ≤ 0.75),0.770,0.781,0.759,0.863


In [19]:
final_models = {
    "Logistic regression (C=0.1)": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000)),
    "Decision tree (max_depth=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "k-NN (k=31)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=31)),
}

predictions = {}   

for name, model in final_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1]
    predictions[name] = (pred, score)
    test_results.append(evaluate(name, y_test, pred, score))

results_table = pd.DataFrame(test_results).round(3)
results_table

,model,F1,Precision,Recall,ROC-AUC
0,Dummy (stratified),0.472,0.473,0.472,0.473
1,Rules (edit dist ≤ 0.75),0.770,0.781,0.759,0.863
2,Logistic regression (C=0.1),0.869,0.897,0.843,0.943
3,Decision tree (max_depth=3),0.849,0.841,0.857,0.916
4,k-NN (k=31),0.875,0.901,0.850,0.938


All results are classification performance on the held-out test set against the constructed Gaois cluster-based labels. All three learned models clearly outperform both baselines (F1 0.85–0.88 vs 0.770 for the rules baseline). k-NN has the highest test F1 (0.875) and logistic regression the highest ROC-AUC (0.943), but the difference between them (0.006 F1) is smaller than the cross-validation standard deviation, so they perform equivalently. Logistic regression is selected as the best model because it was best in cross-validation (model selection was not based on the test set), and it is the simplest and most interpretable. The decision tree has the highest recall but the lowest precision of the learned models.